# L05 · Inverse Kinematics, End-Effector Poses, and Cameras

This lab follows one short chain from a world-frame hand target to measured motion:

```text
current q → FK → current pose
target pose → IK + residual check → q goal → position control → measured pose
scene state → fixed camera → RGB + depth
```

FK predicts; IK proposes; the controller executes; measured state provides the result. The final camera section is deliberately brief.

## Before you run

`ROBO_GENESIS_BACKEND=auto` selects the verified AMD backend when available and otherwise uses CPU. Set it to `cpu` for the minimum path. `ROBO_GENESIS_RENDER=0` runs all FK, IK, control, and unreachable-target checks without creating a camera; set it to `1` before starting the kernel to require one real fixed-camera RGB/depth observation. Restart the kernel before changing either setting.

Predict first:

1. Does a finite `(9,)` q prove that IK converged?
2. Does calling FK for a q move the robot?
3. Does a low IK residual prove that the controller reached the target?
4. If `res=(640, 360)`, what RGB and depth shapes should the camera return?

In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np

from robo_genesis.course_manifest import load_course_manifest
from robo_genesis.course_utils import environment_report, notebook_mode, select_backend, to_numpy
from robo_genesis.scene_config import (
    FRANKA_FORCE_MAX,
    FRANKA_FORCE_MIN,
    FRANKA_KP,
    FRANKA_KV,
    FRANKA_MJCF,
    FRANKA_QPOS,
)

lesson = load_course_manifest().lesson("L05")
assert lesson.slug == "inverse-kinematics-end-effector-poses-and-cameras"
assert lesson.status.value == "cpu-verified"

backend_mode = os.environ.get("ROBO_GENESIS_BACKEND", "auto").strip().lower()
if backend_mode not in {"auto", "cpu"}:
    raise ValueError("ROBO_GENESIS_BACKEND must be 'auto' or 'cpu'")
render_value = os.environ.get("ROBO_GENESIS_RENDER", "0").strip()
if render_value not in {"0", "1"}:
    raise ValueError("ROBO_GENESIS_RENDER must be '0' or '1'")
render_enabled = render_value == "1"

runtime = notebook_mode("l05-ik-poses-cameras", show_viewer=False)
environment = environment_report()

import genesis as gs

backend = gs.cpu if backend_mode == "cpu" else select_backend(prefer_rocm=True)
gs.init(backend=backend, seed=0, precision="32", logging_level="warning")

if getattr(gs, "amdgpu", None) is not None and gs.backend == gs.amdgpu:
    actual_backend = "amdgpu"
elif gs.backend == gs.cpu:
    actual_backend = "cpu"
else:
    actual_backend = str(gs.backend)

print("Genesis:", environment["genesis_world"])
print("requested backend:", backend_mode)
print("actual backend:", actual_backend)
print("render enabled:", render_enabled)
print("output directory:", runtime["output_dir"].resolve())

## Build one scene

The target marker, Franka, and optional fixed camera are declared before `scene.build()`. The camera exists only when rendering was requested; the numerical FK/IK path needs no graphics stack. After build, reuse the nine named DOFs and position-control settings from L04.

In [ ]:
DT = 0.01
SUBSTEPS = 2
REACH_STEPS = 180
IK_POSITION_TOLERANCE = 5e-4
IK_ROTATION_TOLERANCE = 5e-3
TARGET_POSITION = np.array([0.45, 0.0, 0.35], dtype=float)
TARGET_QUATERNION = np.array([0.0, 1.0, 0.0, 0.0], dtype=float)
UNREACHABLE_POSITION = np.array([2.0, 0.0, 2.0], dtype=float)
CAMERA_RESOLUTION = (640, 360)

scene = gs.Scene(
    sim_options=gs.options.SimOptions(dt=DT, substeps=SUBSTEPS),
    show_viewer=False,
)
scene.add_entity(gs.morphs.Plane())
franka = scene.add_entity(gs.morphs.MJCF(file=FRANKA_MJCF))
scene.add_entity(
    gs.morphs.Sphere(
        radius=0.025,
        pos=tuple(TARGET_POSITION),
        fixed=True,
        collision=False,
    ),
    surface=gs.surfaces.Default(color=(0.95, 0.20, 0.20, 1.0)),
)
camera = None
if render_enabled:
    camera = scene.add_camera(
        res=CAMERA_RESOLUTION,
        pos=(1.2, -1.2, 1.0),
        lookat=(0.35, 0.0, 0.35),
        fov=45,
        GUI=False,
    )
scene.build()

hand = franka.get_link("hand")
joint_names = [f"joint{i}" for i in range(1, 8)] + [
    "finger_joint1",
    "finger_joint2",
]
all_dofs = np.asarray(
    [franka.get_joint(name).dofs_idx_local[0] for name in joint_names],
    dtype=int,
)
arm_dofs = all_dofs[:7]
q_start = np.asarray(FRANKA_QPOS, dtype=float)

franka.set_dofs_kp(np.asarray(FRANKA_KP), dofs_idx_local=all_dofs)
franka.set_dofs_kv(np.asarray(FRANKA_KV), dofs_idx_local=all_dofs)
franka.set_dofs_force_range(
    np.asarray(FRANKA_FORCE_MIN),
    np.asarray(FRANKA_FORCE_MAX),
    dofs_idx_local=all_dofs,
)
franka.set_dofs_position(q_start, dofs_idx_local=all_dofs, zero_velocity=True)

print(f"Franka: {franka.n_dofs} DOFs; arm indices={arm_dofs.tolist()}")
print("camera:", "declared before build" if render_enabled else "SKIP")

## Read the pose, then connect IK and FK

The target and measured hand pose are all expressed in the world frame. Genesis uses unit quaternions in w-x-y-z order; `q` and `-q` describe the same rotation. The first code cell below exposes only the small pose and residual helpers needed by the experiment.

Conceptually FK comes first: q predicts a pose. In Genesis 1.3.3, however, `forward_kinematics()` reuses scratch storage initialized by the first IK call, so the executable cell solves IK and then immediately uses FK to inspect that candidate. FK still does not execute motion.

In [ ]:
def normalize_wxyz(quaternion):
    quaternion = np.asarray(quaternion, dtype=float)
    if quaternion.shape != (4,) or not np.isfinite(quaternion).all():
        raise ValueError("expected a finite wxyz quaternion with shape (4,)")
    norm = float(np.linalg.norm(quaternion))
    if norm < 1e-12:
        raise ValueError("a zero quaternion does not define an orientation")
    return quaternion / norm


def quaternion_angle_error(measured_wxyz, target_wxyz):
    measured = normalize_wxyz(measured_wxyz)
    target = normalize_wxyz(target_wxyz)
    cosine_half_angle = np.clip(abs(np.dot(measured, target)), 0.0, 1.0)
    return float(2.0 * np.arccos(cosine_half_angle))


def read_hand_pose():
    position = to_numpy(hand.get_pos(relative=False)).reshape(3).astype(float)
    quaternion = to_numpy(hand.get_quat(relative=False)).reshape(4).astype(float)
    if not np.isfinite(position).all() or not np.isfinite(quaternion).all():
        raise AssertionError("hand pose contains non-finite values")
    return position, quaternion


def summarize_ik(q_raw, error_raw):
    q = to_numpy(q_raw).reshape(-1).astype(float)
    error = to_numpy(error_raw).reshape(-1).astype(float)
    position_residual = float(np.linalg.norm(error[:3]))
    rotation_residual = float(np.linalg.norm(error[3:]))
    valid = bool(
        q.shape == (9,)
        and error.shape == (6,)
        and np.isfinite(q).all()
        and np.isfinite(error).all()
        and position_residual <= IK_POSITION_TOLERANCE
        and rotation_residual <= IK_ROTATION_TOLERANCE
    )
    return q, error, position_residual, rotation_residual, valid


initial_q = to_numpy(
    franka.get_dofs_position(dofs_idx_local=all_dofs)
).reshape(-1).astype(float)
initial_position, initial_quaternion = read_hand_pose()
target_quaternion = normalize_wxyz(TARGET_QUATERNION)

print("initial q shape:", initial_q.shape)
print("initial hand position [m]:", initial_position)
print("initial hand quaternion [wxyz]:", initial_quaternion)
print("target position [m]:", TARGET_POSITION)
print("target quaternion [wxyz]:", target_quaternion)

In [ ]:
q_candidate_raw, ik_error_raw = franka.inverse_kinematics(
    link=hand,
    pos=TARGET_POSITION,
    quat=target_quaternion,
    init_qpos=q_start,
    dofs_idx_local=arm_dofs,
    respect_joint_limit=True,
    pos_tol=IK_POSITION_TOLERANCE,
    rot_tol=IK_ROTATION_TOLERANCE,
    return_error=True,
)
(
    q_candidate,
    ik_error,
    ik_position_residual,
    ik_rotation_residual,
    reachable_ik_valid,
) = summarize_ik(q_candidate_raw, ik_error_raw)

print("IK q shape:", q_candidate.shape)
print("IK error shape:", ik_error.shape)
print("IK position residual [m]:", ik_position_residual)
print("IK rotation residual [rad]:", ik_rotation_residual)
print("IK valid:", reachable_ik_valid)
if not reachable_ik_valid:
    raise AssertionError("reachable IK candidate failed the residual check")

fk_positions_raw, fk_quaternions_raw = franka.forward_kinematics(
    q_candidate_raw,
    links_idx_local=[hand.idx_local],
)
fk_position = to_numpy(fk_positions_raw).reshape(3).astype(float)
fk_quaternion = to_numpy(fk_quaternions_raw).reshape(4).astype(float)
fk_position_error = float(np.linalg.norm(fk_position - TARGET_POSITION))
fk_orientation_error = quaternion_angle_error(fk_quaternion, target_quaternion)

print("FK predicted position [m]:", fk_position)
print("FK predicted quaternion [wxyz]:", fk_quaternion)
print("FK position error [m]:", fk_position_error)
print("FK orientation error [rad]:", fk_orientation_error)

## Execute the accepted q and measure the result

IK and FK have not executed a dynamic motion. The next cell sends the accepted whole-robot q through the same position-control path used in L04, advances the Scene for 180 steps, and reads the world-frame hand pose after every step. The final thresholds apply only to this scene and finite control window.

In [ ]:
position_history = [initial_position]
quaternion_history = [initial_quaternion]
for _ in range(REACH_STEPS):
    franka.control_dofs_position(q_candidate, dofs_idx_local=all_dofs)
    scene.step()
    position, quaternion = read_hand_pose()
    position_history.append(position)
    quaternion_history.append(quaternion)

position_history = np.asarray(position_history)
quaternion_history = np.asarray(quaternion_history)
time = np.arange(REACH_STEPS + 1) * DT
position_errors = np.linalg.norm(position_history - TARGET_POSITION, axis=1)
orientation_errors = np.asarray(
    [quaternion_angle_error(q, target_quaternion) for q in quaternion_history]
)
final_position_error = float(position_errors[-1])
final_orientation_error = float(orientation_errors[-1])

execution_checks = {
    "trajectory_finite": (
        np.isfinite(position_history).all()
        and np.isfinite(quaternion_history).all()
    ),
    "position_error_decreased": final_position_error < position_errors[0],
    "orientation_improved_or_initially_satisfied": (
        orientation_errors[0] <= IK_ROTATION_TOLERANCE
        or final_orientation_error < orientation_errors[0]
    ),
    "position_threshold": final_position_error < 0.02,
    "orientation_threshold": final_orientation_error < 0.05,
}
if not all(execution_checks.values()):
    raise AssertionError(execution_checks)

print("final measured position [m]:", position_history[-1])
print("final measured quaternion [wxyz]:", quaternion_history[-1])
print("final position error [m]:", final_position_error)
print("final orientation error [rad]:", final_orientation_error)

figure, axes = plt.subplots(1, 2, figsize=(10, 3.5))
axes[0].plot(time, position_errors)
axes[0].axhline(0.02, color="tab:red", linestyle="--")
axes[0].set(xlabel="time [s]", ylabel="position error [m]")
axes[1].plot(time, orientation_errors)
axes[1].axhline(0.05, color="tab:red", linestyle="--")
axes[1].set(xlabel="time [s]", ylabel="orientation error [rad]")
for axis in axes:
    axis.grid(alpha=0.25)
figure.suptitle("Measured end-effector pose errors")
figure.tight_layout()
plt.show()

## One fixed camera

The camera is a brief bridge to later visual observations. When rendering is enabled, acquire one RGB/depth pair and check the direct array contract: camera configuration uses `res=(W, H)`, while returned arrays use `(H, W, 3)` and `(H, W)`. A camera image can show the scene, but the numerical errors above remain the evidence for IK and control.

In [ ]:
camera_path_ok = True
camera_status = "SKIP — ROBO_GENESIS_RENDER=0; no camera was created"
if render_enabled:
    rgb, depth, _, _ = camera.render(rgb=True, depth=True)
    rgb = to_numpy(rgb)
    depth = to_numpy(depth)
    width, height = CAMERA_RESOLUTION
    camera_checks = {
        "rgb_shape": rgb.shape == (height, width, 3),
        "depth_shape": depth.shape == (height, width),
        "rgb_dtype": rgb.dtype == np.uint8,
        "depth_float": np.issubdtype(depth.dtype, np.floating),
        "pixels_finite": np.isfinite(rgb).all() and np.isfinite(depth).all(),
    }
    camera_path_ok = all(camera_checks.values())
    if not camera_path_ok:
        raise AssertionError(camera_checks)
    camera_status = "PASSED — fixed-camera RGB/depth captured"
    print("RGB:", rgb.shape, rgb.dtype)
    print("depth:", depth.shape, depth.dtype)

    figure, axes = plt.subplots(1, 2, figsize=(10, 4))
    axes[0].imshow(rgb)
    axes[0].set_title("Fixed camera — RGB")
    axes[1].imshow(depth, cmap="viridis")
    axes[1].set_title("Fixed camera — depth [m]")
    for axis in axes:
        axis.axis("off")
    figure.tight_layout()
    plt.show()

print(camera_status)

## Reject an unreachable target

Run the same IK and residual check for `[2, 0, 2] m`. Genesis may still return a finite best-effort q. The correct normal path is to reject it when either residual exceeds tolerance and send no control command. This negative case is enough; the notebook does not execute rejected q.

In [ ]:
q_unreachable_raw, unreachable_error_raw = franka.inverse_kinematics(
    link=hand,
    pos=UNREACHABLE_POSITION,
    quat=target_quaternion,
    init_qpos=q_start,
    dofs_idx_local=arm_dofs,
    respect_joint_limit=True,
    pos_tol=IK_POSITION_TOLERANCE,
    rot_tol=IK_ROTATION_TOLERANCE,
    return_error=True,
)
(
    q_unreachable,
    unreachable_error,
    unreachable_position_residual,
    unreachable_rotation_residual,
    unreachable_ik_valid,
) = summarize_ik(q_unreachable_raw, unreachable_error_raw)
unreachable_command_sent = False

print("unreachable q finite:", np.isfinite(q_unreachable).all())
print("unreachable position residual [m]:", unreachable_position_residual)
print("unreachable rotation residual [rad]:", unreachable_rotation_residual)
print("IK valid:", unreachable_ik_valid)
print("command sent:", "yes" if unreachable_command_sent else "no")

final_checks = {
    "genesis_1_3_3": environment["genesis_world"] == "1.3.3",
    "actual_backend_supported": actual_backend in {"cpu", "amdgpu"},
    "forced_cpu_honored": backend_mode != "cpu" or actual_backend == "cpu",
    "model_shape": initial_q.shape == (9,) and arm_dofs.shape == (7,),
    "reachable_ik_accepted": reachable_ik_valid,
    "fk_position_matches_target": fk_position_error <= IK_POSITION_TOLERANCE,
    "fk_orientation_matches_target": fk_orientation_error <= IK_ROTATION_TOLERANCE,
    "dynamic_execution": all(execution_checks.values()),
    "camera_branch": camera_path_ok,
    "unreachable_rejected": not unreachable_ik_valid,
    "unreachable_not_commanded": not unreachable_command_sent,
}
for name, passed in final_checks.items():
    print(f"{'PASS' if passed else 'FAIL'} — {name}")
failed = [name for name, passed in final_checks.items() if not passed]
if failed:
    raise AssertionError("L05 checks failed: " + ", ".join(failed))

print("camera:", camera_status)
print("L05 CHECK: PASSED")

## Checkpoint and next step

Use the values printed by this run to answer:

1. What are the inputs and outputs of FK?
2. What are the inputs and outputs of IK?
3. Why are IK residual, FK prediction error, and measured execution error different evidence?
4. Why was the unreachable candidate rejected even if its q was finite?
5. What did the camera path prove, or what was explicitly skipped?

As a small exercise, increase only the reachable target x coordinate by `0.03 m`, predict whether it remains reachable, and compare the new IK residual, FK prediction, and measured final error. Do not add a table, object, batch dimension, or another camera.

L06 will apply the same IK and control contract across parallel environments. L07 will place the controlled robot into a grasping scene, and L08 will sequence multiple pose targets with safe motion and grasp logic rather than treating IK as a path planner.